# Pseudobulk DESeq2 — Parameterised DGE Notebook

Set `COMPARISON` below to any key from `dge_comparisons.yaml`, then run all cells.

**Environment:** `pydeseq-env`

In [ ]:
# ── Configuration ──
# Change this to run a different comparison:
COMPARISON = "d2_lapa_iscs"

In [ ]:
import sys
from pathlib import Path
import yaml

sys.path.insert(0, str(Path("../.." ).resolve()))

from src.config import DATASETS, DGE_OUTPUT_DIR
from src.dge import create_pseudobulk, merge_pseudobulk, build_clinical_df, run_pydeseq2, prepare_for_volcanoser

import scanpy as sc
import pandas as pd

In [ ]:
# ── Load comparison config ──
with open("dge_comparisons.yaml") as f:
    all_comparisons = yaml.safe_load(f)

cfg = all_comparisons[COMPARISON]
print(f"Running: {cfg['title']}")
print(f"  Cell type:  {cfg['cell_type']}")
print(f"  Datasets:   {cfg['datasets']}")
print(f"  Contrast:   {cfg['contrast']}")

In [ ]:
# ── Create pseudobulk per dataset ──
pb_list = []
for ds_key in cfg["datasets"]:
    ds = DATASETS[ds_key]
    adata = sc.read_h5ad(ds["labelled"])
    pb = create_pseudobulk(adata, cell_type=cfg["cell_type"])
    pb_list.append(pb)
    del adata  # free memory

In [ ]:
# ── Merge and prepare for DESeq2 ──
pb_merged = merge_pseudobulk(*pb_list)
counts_df = pb_merged.T  # samples x genes

clinical_df = build_clinical_df(counts_df, cfg["conditions_map"])
print(clinical_df)

In [ ]:
# ── Run DESeq2 ──
results = run_pydeseq2(counts_df, clinical_df, contrast=cfg["contrast"])
results.head(20)

In [ ]:
# ── Save results ──
DGE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
out_path = DGE_OUTPUT_DIR / f"{COMPARISON}_deseq2_results.csv"
results.to_csv(out_path)
print(f"Saved: {out_path}")

In [ ]:
# ── Summary statistics ──
sig = results.dropna(subset=["padj"])
n_up = ((sig["padj"] < 0.05) & (sig["log2FoldChange"] > 1)).sum()
n_down = ((sig["padj"] < 0.05) & (sig["log2FoldChange"] < -1)).sum()
print(f"\nSignificant DEGs (padj<0.05, |LFC|>1):")
print(f"  Upregulated:   {n_up}")
print(f"  Downregulated: {n_down}")

In [ ]:
# ── Prepare for Volcanoser ──
volcanoser_df = prepare_for_volcanoser(results)
volcanoser_path = DGE_OUTPUT_DIR / f"{COMPARISON}_deseq2_volcanoser.csv"
volcanoser_df.to_csv(volcanoser_path)
print(f"Volcanoser-ready file saved: {volcanoser_path}")
volcanoser_df.head()